In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
url = 'https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=24'

In [3]:
bestseller = requests.get(url)
bestseller.text[:500]

'\r\n\r\n\r\n\r\n\r\n    \r\n\t<!DOCTYPE html >\r\n\t<html lang="ko">\r\n\r\n<head><link rel="alternate" media="only screen and(max-width: 640px)" href="https://m.yes24.com/Home/Best?DispNo=001">\r\n\t<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\r\n\t<meta http-equiv="Content-Type" content="text/html;charset=utf-8" />\r\n\t<meta http-equiv="Accept-CH" content="dpr, width, viewport-width, rtt, downlink, ect, UA, UA-Platform, UA-Arch, UA-Model, UA-Mobile, UA-Full-Version" />\r\n\t<meta http-equiv="Accept-CH-Lifetime" c'

In [ ]:
selector = '#yesBestList div.item_info'
soup = BeautifulSoup(bestseller.text,'html.parser')
items=soup.select(selector)
len(items) # 베스트셀러 상품 수
items[0] # 첫번째 상품의 HTML 코드

In [5]:
title_selector='div.info_name > a' # a.gd_name'
items[23].select_one(title_selector).text

'더블 클릭'

In [6]:
auth_selector='span.info_auth > a'
items[23].select_one(auth_selector).text

'알간지'

In [7]:
price_selector = 'div.info_price > strong'
items[23].select_one(price_selector).text

'17,100원'

In [ ]:
#img_selector = 'span.img_grp'
#items[23].select_one(img_selector)

# 한 페이지의 도서 리스트 추출

In [15]:
scraped_data = [] # 추출한 데이터를 담을 빈 리스트

title_selector = 'div.info_name > a'
auth_selector = 'span.info_auth > a'
price_selector = 'div.info_price > strong'

for i in range(24):
    title = items[i].select_one(title_selector).text.strip()
    auth = items[i].select_one(auth_selector).text.strip()
    price = items[i].select_one(price_selector).text.strip()
    
    item_info = {
        '제목': title,
        '저자': auth,
        '가격': price
    }
    
    scraped_data.append(item_info)

print(scraped_data)

[{'제목': '나의 완벽한 장례식', '저자': '조현선', '가격': '17,100원'}, {'제목': '진보를 위한 주식투자', '저자': '이광수', '가격': '19,800원'}, {'제목': '박곰희 연금 부자 수업', '저자': '박곰희', '가격': '18,900원'}, {'제목': '괴테는 모든 것을 말했다', '저자': '스즈키 유이', '가격': '15,300원'}, {'제목': 'ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC', '저자': 'ETS', '가격': '19,800원'}, {'제목': 'ETS 토익 정기시험 기출문제집 1000 Vol. 5 LC', '저자': 'ETS', '가격': '19,800원'}, {'제목': '죽은 왕녀를 위한 파반느 (양장 특별판)', '저자': '박민규', '가격': '18,000원'}, {'제목': '자몽살구클럽', '저자': '한로로', '가격': '10,800원'}, {'제목': '2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상', '저자': '최태성', '가격': '17,100원'}, {'제목': '너를 아끼며 살아라', '저자': '나태주', '가격': '16,650원'}, {'제목': '모순', '저자': '양귀자', '가격': '11,700원'}, {'제목': '돈의 방정식', '저자': '모건 하우절', '가격': '25,200원'}, {'제목': '자본주의 시대에서 살아남기 위한 최소한의 경제 공부', '저자': '백억남(김욱현)', '가격': '25,200원'}, {'제목': '1,000만 원으로 3년 안에 300만 원 월배당 만들기', '저자': '인생업(임승현)', '가격': '20,700원'}, {'제목': '위버멘쉬', '저자': '프리드리히 니체', '가격': '16,020원'}, {'제목': '엄마가 유령이 되었어!', '저자': '노부미', '가격': '10,800원'}, {'제목': '안녕이라 그랬어', '저자': '김애란', '

In [17]:
import pandas as pd

df = pd.DataFrame(scraped_data)
df.to_csv("bestseller_data.csv", index=False, encoding='utf-8-sig')
print("판다스를 이용해 CSV 파일 저장이 완료되었습니다!")

판다스를 이용해 CSV 파일 저장이 완료되었습니다!


# 여러 페이지의 도서 리스트 추출
# Book 클래스를 이용한 객체 생성

In [27]:
# Book 클래스 정의 (코드구조상 가장 위에)
class Book:
    def __init__(self, title, author, price):
        self.title = title
        self.author = author
        self.price = price
        
    def __str__(self):
        return f"[{self.title}] - {self.author} 지음 ({self.price})"

book_objects = [] # 생성된 Book 객체들을 담을 빈 리스트

# 크롤링 선택자 정의
title_selector = 'div.info_name > a'
auth_selector = 'span.info_auth > a'
price_selector = 'div.info_price > strong'
item_container_selector = '.itemUnit'

# 여러 페이지를 크롤링 및 객체 생성 시작
for page in range(1, 7):
    url = f'https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber={page}&pageSize=24'
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    items = soup.select(item_container_selector)
    
    for i in range(len(items)): 
        title_elem = items[i].select_one(title_selector)
        auth_elem = items[i].select_one(auth_selector)
        price_elem = items[i].select_one(price_selector)
        
        title = title_elem.text.strip() if title_elem else '제목 없음'
        auth = auth_elem.text.strip() if auth_elem else '저자 없음'
        price = price_elem.text.strip() if price_elem else '가격 없음'
        book = Book(title, auth, price)  # 추출한 데이터로 즉시 Book 객체를 생성
    
        book_objects.append(book) # 생성된 객체를 리스트에 보관
        
        print(f"{(page-1)*24 + i+1}. {book}") # book 객체를 그대로 print()에 넣으면 클래스 안 __str__ 함수가 작동해 예쁘게 출력
        
    time.sleep(1) # 서버 차단 방지

print(f"\n🎉 크롤링 및 객체 생성 완료! 총 {len(book_objects)}개의 Book 객체가 리스트에 담겼습니다.")

1. [나의 완벽한 장례식] - 조현선 지음 (17,100원)
2. [진보를 위한 주식투자] - 이광수 지음 (19,800원)
3. [박곰희 연금 부자 수업] - 박곰희 지음 (18,900원)
4. [괴테는 모든 것을 말했다] - 스즈키 유이 지음 (15,300원)
5. [ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC] - ETS 지음 (19,800원)
6. [ETS 토익 정기시험 기출문제집 1000 Vol. 5 LC] - ETS 지음 (19,800원)
7. [죽은 왕녀를 위한 파반느 (양장 특별판)] - 박민규 지음 (18,000원)
8. [자몽살구클럽] - 한로로 지음 (10,800원)
9. [2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상] - 최태성 지음 (17,100원)
10. [너를 아끼며 살아라] - 나태주 지음 (16,650원)
11. [모순] - 양귀자 지음 (11,700원)
12. [돈의 방정식] - 모건 하우절 지음 (25,200원)
13. [자본주의 시대에서 살아남기 위한 최소한의 경제 공부] - 백억남(김욱현) 지음 (25,200원)
14. [1,000만 원으로 3년 안에 300만 원 월배당 만들기] - 인생업(임승현) 지음 (20,700원)
15. [위버멘쉬] - 프리드리히 니체 지음 (16,020원)
16. [엄마가 유령이 되었어!] - 노부미 지음 (10,800원)
17. [안녕이라 그랬어] - 김애란 지음 (15,120원)
18. [2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하] - 최태성 지음 (16,650원)
19. [최소한의 삼국지] - 최태성 지음 (17,550원)
20. [싯다르타] - 헤르만 헤세 지음 (7,200원)
21. [2026 심우철 실전 동형 모의고사 Season 2] - 심우철 지음 (13,500원)
22. [사이토 히토리의 어떻게 살 것인가] - 사이토 히토리 지음 (11,700원)
23. [돈의 심리학 (50만 부 기념 뉴 에디션)]

# SQLite db(내장db) 저장

In [32]:
import sqlite3

# 데이터베이스 연결 및 커서(Cursor) 생성
conn = sqlite3.connect('bestsellers.db')
cursor = conn.cursor()

# 테이블 생성 
# books라는 이름의 테이블
cursor.execute('''
    CREATE TABLE IF NOT EXISTS books (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT,
        author TEXT,
        price TEXT
    )
''')

# 데이터 삽입 
# Book 객체들이 담긴 book_objects 리스트를 순회하며 데이터를 DB에 넣음
# executemany를 사용하기 위해 Book 객체의 속성들을 튜플(tuple) 형태의 리스트로 변환
book_data_list = [(book.title, book.author, book.price) for book in book_objects]

cursor.executemany('''
    INSERT INTO books (title, author, price)
    VALUES (?, ?, ?)  
''', book_data_list) # ? 기호를 사용해 데이터를 안전하게 일괄 삽입

conn.commit() # 변경사항을 DB에 영구적으로 저장
print(f"총 {cursor.rowcount}개의 데이터가 DB에 성공적으로 저장되었습니다!\n")


# 데이터 조회
print("--- DB에서 데이터 조회해보기 (상위 5개) ---")
cursor.execute('SELECT * FROM books LIMIT 5') # books 테이블에서 모든 데이터(*)를 가져오되, 5개(LIMIT 5)만
rows = cursor.fetchall() # 조회한 데이터를 모두 가져와 rows 변수에 담음

# 가져온 데이터를 하나씩 출력
for row in rows:  # row는 (id, title, author, price) 형태의 튜플
    print(f"ID: {row[0]} | 제목: {row[1]} | 저자: {row[2]} | 가격: {row[3]}")

# 데이터베이스 연결 종료 (작업이 끝나면 반드시 닫아주어야 합니다)
conn.close()

총 144개의 데이터가 DB에 성공적으로 저장되었습니다!

--- DB에서 데이터 조회해보기 (상위 5개) ---
ID: 1 | 제목: 나의 완벽한 장례식 | 저자: 조현선 | 가격: 17,100원
ID: 2 | 제목: 진보를 위한 주식투자 | 저자: 이광수 | 가격: 19,800원
ID: 3 | 제목: 박곰희 연금 부자 수업 | 저자: 박곰희 | 가격: 18,900원
ID: 4 | 제목: 괴테는 모든 것을 말했다 | 저자: 스즈키 유이 | 가격: 15,300원
ID: 5 | 제목: ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC | 저자: ETS | 가격: 19,800원


# (참고) mysql 저장

In [33]:
import pymysql
conn = pymysql.connect(
  host='localhost', 
  user='root',
  password='1234',
  db='book_db',
  charset='utf8mb4'
)
conn

In [35]:
with conn.cursor() as cursor:
  cursor.execute('''
    CREATE TABLE IF NOT EXISTS books (
        id INT AUTO_INCREMENT PRIMARY KEY,
        title VARCHAR(255),
        author VARCHAR(255),
        price VARCHAR(50)
    )
''')

In [ ]:
with conn.cursor() as cursor:
  for book in book_objects:
    cursor.execute('''
                   INSERT INTO books(title,author, price) VALUES (%s,%s,%s)
                   ''',(book.title, book.author, book.price))
    
  